# SmoRestor Style Desmoking Training

이 노트북은 수술용 내시경 이미지의 연기를 제거(Desmoking)하는 모델을 학습하고 시각화하는 과정을 담고 있습니다.
이전에 모듈화했던 파이썬 스크립트의 기능들을 하나의 노트북으로 모아, 진행 과정을 실시간으로 모니터링할 수 있도록 구성했습니다.

## 1. 환경 설정 및 패키지 불러오기

In [ ]:
import random
from pathlib import Path
import time
import csv
from dataclasses import dataclass
from typing import Dict, List, Sequence, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.io import read_image
from torchvision.utils import make_grid

import numpy as np
import cv2
from skimage import color
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# 재현성을 위한 시드 고정
def seed_everything(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
seed_everything()

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. 유틸리티 함수 및 평가지표

In [ ]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def psnr(pred: torch.Tensor, target: torch.Tensor, max_val: float = 1.0) -> torch.Tensor:
    mse = F.mse_loss(pred, target)
    if mse == 0:
        return torch.tensor(100.0, device=pred.device)
    return 20 * torch.log10(torch.tensor(max_val, device=pred.device)) - 10 * torch.log10(mse + 1e-8)

def ssim_simple(pred: torch.Tensor, target: torch.Tensor, window_size: int = 11) -> torch.Tensor:
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    pad = window_size // 2

    mu_x = F.avg_pool2d(pred, window_size, 1, pad)
    mu_y = F.avg_pool2d(target, window_size, 1, pad)

    sigma_x = F.avg_pool2d(pred * pred, window_size, 1, pad) - mu_x * mu_x
    sigma_y = F.avg_pool2d(target * target, window_size, 1, pad) - mu_y * mu_y
    sigma_xy = F.avg_pool2d(pred * target, window_size, 1, pad) - mu_x * mu_y

    num = (2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)
    den = (mu_x * mu_x + mu_y * mu_y + c1) * (sigma_x + sigma_y + c2)
    return (num / (den + 1e-8)).mean()

def calculate_fade(image_tensor: torch.Tensor) -> float:
    if image_tensor is None:
        return 0.0
    fade_scores = []
    for img in image_tensor:
        img_np = img.detach().cpu().permute(1, 2, 0).numpy()
        dark_channel = np.min(img_np, axis=2)
        dc_mean = np.mean(dark_channel)
        hsv = color.rgb2hsv(img_np)
        saturation = np.mean(hsv[:, :, 1])
        score = dc_mean / (saturation + 1e-6)
        fade_scores.append(score)
    return float(np.mean(fade_scores))

def calculate_ciede2000(pred: torch.Tensor, target: torch.Tensor) -> float:
    pred_np = pred.detach().cpu().permute(0, 2, 3, 1).numpy()
    target_np = target.detach().cpu().permute(0, 2, 3, 1).numpy()
    ciede_scores = []
    for p, t in zip(pred_np, target_np):
        p_lab = color.rgb2lab(p)
        t_lab = color.rgb2lab(t)
        delta_e = color.deltaE_ciede2000(p_lab, t_lab)
        ciede_scores.append(np.mean(delta_e))
    return float(np.mean(ciede_scores))

def write_metrics_csv(path: Path, metrics: Dict[str, float]) -> None:
    ensure_dir(path.parent)
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        for k, v in metrics.items():
            writer.writerow([k, v])

## 3. 데이터셋 구성
데이터 폴더에서 원본(연기) 이미지와 GT(정답) 이미지를 쌍으로 묶어 로드합니다.

In [ ]:
def load_rgb(path: Path) -> torch.Tensor:
    img = read_image(str(path)).float() / 255.0
    if img.shape[0] == 1:
        img = img.repeat(3, 1, 1)
    elif img.shape[0] > 3:
        img = img[:3]
    return img

def discover_pairs(data_dir: Path) -> List[Tuple[Path, Path, str]]:
    if not data_dir.exists():
        raise FileNotFoundError(f"디렉토리를 찾을 수 없습니다: {data_dir}")

    all_png = sorted(data_dir.glob("*.png"))
    gt_map = {}
    smoky_map = {}

    for p in all_png:
        stem = p.stem
        if stem.endswith("_gt"):
            gt_map[stem[:-3]] = p
        else:
            smoky_map[stem] = p

    ids = sorted(set(smoky_map.keys()) & set(gt_map.keys()), key=lambda x: int(x) if x.isdigit() else x)
    pairs = [(smoky_map[i], gt_map[i], i) for i in ids]
    
    if not pairs:
        raise FileNotFoundError(f"'{data_dir}'에서 1.png와 1_gt.png 형태의 파일 쌍을 찾지 못했습니다.")
    return pairs

def split_pairs(pairs: Sequence[Tuple[Path, Path, str]], train_ratio=0.8, val_ratio=0.1, seed=42):
    pairs = list(pairs)
    rng = random.Random(seed)
    rng.shuffle(pairs)

    n = len(pairs)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    n_test = n - n_train - n_val

    return pairs[:n_train], pairs[n_train:n_train + n_val], pairs[n_train + n_val:]

class FlatPairDataset(Dataset):
    def __init__(self, pairs, img_size=(352, 704), augment=False):
        self.pairs = list(pairs)
        self.resize = transforms.Resize(img_size, antialias=True)
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        smoky_path, clean_path, sample_id = self.pairs[idx]
        smoky = self.resize(load_rgb(smoky_path))
        clean = self.resize(load_rgb(clean_path))

        if self.augment:
            if random.random() < 0.5:
                smoky = torch.flip(smoky, dims=[2])
                clean = torch.flip(clean, dims=[2])
            if random.random() < 0.5:
                smoky = torch.flip(smoky, dims=[1])
                clean = torch.flip(clean, dims=[1])

        return {"smoky": smoky, "clean": clean, "name": sample_id}

## 4. 손실 함수 (Loss Function)
L1 + SSIM + VGG Perceptual Loss를 조합하여 화질을 개선합니다.

In [ ]:
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        weights = models.VGG16_Weights.DEFAULT
        feats = models.vgg16(weights=weights).features[:16].eval()
        for p in feats.parameters():
            p.requires_grad = False
        self.features = feats
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406])[None, :, None, None])
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225])[None, :, None, None])

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred_n = (pred - self.mean) / self.std
        target_n = (target - self.mean) / self.std
        return F.l1_loss(self.features(pred_n), self.features(target_n))

@dataclass
class LossWeights:
    l1: float = 1.0
    ssim: float = 0.25
    perceptual: float = 0.05

def color_loss(pred, target):
    pred_mean = torch.mean(pred, dim=(2, 3))
    target_mean = torch.mean(target, dim=(2, 3))
    return F.mse_loss(pred_mean, target_mean)

class DesmokeLoss(nn.Module):
    def __init__(self, w: LossWeights):
        super().__init__()
        self.w = w
        self.perc = VGGPerceptualLoss()

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
        l1 = F.l1_loss(pred, target)
        ssim_loss = 1.0 - ssim_simple(pred, target)
        perc = self.perc(pred, target)
        c_loss = color_loss(pred, target)

        total = self.w.l1 * l1 + self.w.ssim * ssim_loss + self.w.perceptual * perc + 2.0 * c_loss
        stats = {
            "l1": float(l1.detach().cpu()),
            "ssim_loss": float(ssim_loss.detach().cpu()),
            "perc": float(perc.detach().cpu()),
            "color": float(c_loss.detach().cpu()),
            "total": float(total.detach().cpu()),
        }
        return total, stats

## 5. 모델 구조 (SmoRestor-like Model)

In [ ]:
class ConvNormAct(nn.Module):
    def __init__(self, cin: int, cout: int, k: int = 3, s: int = 1, p: Optional[int] = None):
        super().__init__()
        p = k // 2 if p is None else p
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, k, s, p),
            nn.GroupNorm(8, cout),
            nn.GELU(),
        )
    def forward(self, x): return self.block(x)

class ResidualConvBlock(nn.Module):
    def __init__(self, c: int):
        super().__init__()
        self.conv1 = ConvNormAct(c, c)
        self.conv2 = nn.Sequential(nn.Conv2d(c, c, 3, 1, 1), nn.GroupNorm(8, c))
    def forward(self, x): return F.gelu(self.conv2(self.conv1(x)) + x)

class FourierPrior(nn.Module):
    def __init__(self, cutoff_ratio: float = 0.15):
        super().__init__()
        self.cutoff_ratio = cutoff_ratio

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        b, c, h, w = x.shape
        fft = torch.fft.fft2(x, norm="ortho")
        fft = torch.fft.fftshift(fft, dim=(-2, -1))
        yy, xx = torch.meshgrid(torch.linspace(-1, 1, h, device=x.device), torch.linspace(-1, 1, w, device=x.device), indexing="ij")
        rr = torch.sqrt(xx ** 2 + yy ** 2)
        low_mask = (rr <= self.cutoff_ratio).float()[None, None]
        high_mask = 1.0 - low_mask
        low_fft = fft * low_mask
        high_fft = fft * high_mask
        low = torch.fft.ifft2(torch.fft.ifftshift(low_fft, dim=(-2, -1)), norm="ortho").real
        high = torch.fft.ifft2(torch.fft.ifftshift(high_fft, dim=(-2, -1)), norm="ortho").real
        return low, high

class WindowAttention(nn.Module):
    def __init__(self, dim: int, heads: int = 4, window_size: int = 8):
        super().__init__()
        self.dim = dim
        self.heads = heads
        self.window_size = window_size
        self.scale = (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        ws = self.window_size
        pad_h = (ws - h % ws) % ws
        pad_w = (ws - w % ws) % ws
        x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        _, _, hp, wp = x_pad.shape

        xw = x_pad.view(b, c, hp // ws, ws, wp // ws, ws)
        xw = xw.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, ws * ws, c)

        qkv = self.qkv(xw)
        q, k, v = qkv.chunk(3, dim=-1)
        head_dim = c // self.heads
        q = q.view(-1, ws * ws, self.heads, head_dim).transpose(1, 2)
        k = k.view(-1, ws * ws, self.heads, head_dim).transpose(1, 2)
        v = v.view(-1, ws * ws, self.heads, head_dim).transpose(1, 2)

        attn = torch.softmax((q @ k.transpose(-2, -1)) * self.scale, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(-1, ws * ws, c)
        out = self.proj(out)
        out = out.view(b, hp // ws, wp // ws, ws, ws, c)
        out = out.permute(0, 5, 1, 3, 2, 4).contiguous().view(b, c, hp, wp)
        return out[:, :, :h, :w]

class FrequencyBlock(nn.Module):
    def __init__(self, dim: int, heads: int = 4, window_size: int = 8, mlp_ratio: float = 2.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, dim)
        self.fourier = FourierPrior(0.15)
        self.low_proj = nn.Conv2d(dim, dim, 1)
        self.high_proj = nn.Conv2d(dim, dim, 1)
        self.attn = WindowAttention(dim, heads=heads, window_size=window_size)
        hidden = int(dim * mlp_ratio)
        self.norm2 = nn.GroupNorm(8, dim)
        self.ffn = nn.Sequential(nn.Conv2d(dim, hidden, 1), nn.GELU(), nn.Conv2d(hidden, dim, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.norm1(x)
        low, high = self.fourier(y)
        y = self.low_proj(low) + self.high_proj(high)
        y = self.attn(y)
        x = x + y
        x = x + self.ffn(self.norm2(x))
        return x

class Encoder(nn.Module):
    def __init__(self, in_ch: int = 3, base_dim: int = 48):
        super().__init__()
        self.stem = nn.Sequential(ConvNormAct(in_ch, base_dim), ResidualConvBlock(base_dim))
        self.down1 = nn.Sequential(ConvNormAct(base_dim, base_dim * 2, k=3, s=2), ResidualConvBlock(base_dim * 2))
        self.down2 = nn.Sequential(ConvNormAct(base_dim * 2, base_dim * 4, k=3, s=2), ResidualConvBlock(base_dim * 4))
    def forward(self, x): return self.stem(x), self.down1(self.stem(x)), self.down2(self.down1(self.stem(x)))
    def forward(self, x):
        f1 = self.stem(x)
        f2 = self.down1(f1)
        f3 = self.down2(f2)
        return f1, f2, f3

class Decoder(nn.Module):
    def __init__(self, base_dim: int = 48, out_ch: int = 3):
        super().__init__()
        self.up1_t = nn.ConvTranspose2d(base_dim * 4, base_dim * 2, 2, 2)
        self.up1 = nn.Sequential(ConvNormAct(base_dim * 4, base_dim * 2), ResidualConvBlock(base_dim * 2))
        self.up2_t = nn.ConvTranspose2d(base_dim * 2, base_dim, 2, 2)
        self.up2 = nn.Sequential(ConvNormAct(base_dim * 2, base_dim), ResidualConvBlock(base_dim))
        self.head = nn.Conv2d(base_dim, out_ch, 3, 1, 1)

    def forward(self, f1, f2, f3):
        x = self.up1_t(f3)
        x = torch.cat([x, f2], dim=1)
        x = self.up1(x)
        x = self.up2_t(x)
        x = torch.cat([x, f1], dim=1)
        x = self.up2(x)
        return torch.sigmoid(self.head(x))

class SmoRestorLike(nn.Module):
    def __init__(self, base_dim=48, num_blocks=4, heads=4, window_size=8):
        super().__init__()
        self.encoder = Encoder(3, base_dim)
        self.blocks = nn.Sequential(*[FrequencyBlock(base_dim * 4, heads=heads, window_size=window_size) for _ in range(num_blocks)])
        self.decoder = Decoder(base_dim, 3)
    def forward(self, x):
        f1, f2, f3 = self.encoder(x)
        z = self.blocks(f3)
        out = self.decoder(f1, f2, z)
        return torch.clamp(x + (out - 0.5), 0, 1)

## 6. 시각화 및 학습 엔진 구성
학습 중간중간 모델이 어떻게 복원하고 있는지 시각적으로 확인하기 위한 함수를 정의합니다.

In [ ]:
def show_images(smoky, clean, pred, epoch):
    # 배치에서 첫 번째 이미지만 시각화
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # tensor를 이미지로 변환하기 위해 차원 변경 (C, H, W) -> (H, W, C)
    axes[0].imshow(smoky[0].cpu().permute(1, 2, 0).numpy())
    axes[0].set_title("Input (Smoky)")
    axes[0].axis('off')
    
    axes[1].imshow(pred[0].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy())
    axes[1].set_title(f"Prediction (Epoch {epoch})")
    axes[1].axis('off')
    
    axes[2].imshow(clean[0].cpu().permute(1, 2, 0).numpy())
    axes[2].set_title("Ground Truth (Clean)")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

def train_one_epoch(model, loader, criterion, optimizer, epoch):
    model.train()
    agg = {}
    pbar = tqdm(loader, desc=f"Epoch {epoch} Train", leave=False)
    
    for i, batch in enumerate(pbar):
        smoky = batch["smoky"].to(device, non_blocking=True)
        clean = batch["clean"].to(device, non_blocking=True)

        pred = model(smoky)
        loss, stats = criterion(pred, clean)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        for k, v in stats.items():
            agg[k] = agg.get(k, 0.0) + v
            
        pbar.set_postfix({'loss': f"{stats['total']:.4f}"})

        # 배치의 첫 번째 스텝에서만 시각화 수행
        if i == 0:
            show_images(smoky, clean, pred, epoch)

    for k in agg:
        agg[k] /= max(len(loader), 1)
    return agg

@torch.no_grad()
def evaluate(model, loader, epoch):
    model.eval()
    psnr_vals, ssim_vals, fade_vals, ciede_vals = [], [], [], []
    pbar = tqdm(loader, desc=f"Epoch {epoch} Eval", leave=False)

    for batch in pbar:
        smoky = batch["smoky"].to(device, non_blocking=True)
        clean = batch["clean"].to(device, non_blocking=True)

        pred = model(smoky).clamp(0, 1)
        psnr_vals.append(float(psnr(pred, clean).cpu()))
        ssim_vals.append(float(ssim_simple(pred, clean).cpu()))
        fade_vals.append(calculate_fade(pred))
        ciede_vals.append(calculate_ciede2000(pred, clean))

    return {
        "psnr": sum(psnr_vals) / max(len(psnr_vals), 1),
        "ssim": sum(ssim_vals) / max(len(ssim_vals), 1),
        "fade": sum(fade_vals) / max(len(fade_vals), 1),
        "ciede": sum(ciede_vals) / max(len(ciede_vals), 1),
    }

## 7. 하이퍼파라미터 설정 및 학습 실행

In [ ]:
# ----------------- 설정 -----------------
DATA_DIR = Path("assets/DesmokeData_dataset")
OUT_DIR = Path("runs/smore_notebook")
EPOCHS = 20
BATCH_SIZE = 4
IMG_SIZE = (256, 512) # 메모리를 위해 사이즈 약간 축소
LR = 2e-4
NUM_WORKERS = 0 # GPU 학습 시 CPU 병목을 막기 위함 (노트북 환경)

ensure_dir(OUT_DIR)

# ----------------- 로더 준비 -----------------
try:
    pairs = discover_pairs(DATA_DIR)
    train_pairs, val_pairs, test_pairs = split_pairs(pairs, 0.8, 0.1)

    train_loader = DataLoader(FlatPairDataset(train_pairs, IMG_SIZE, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(FlatPairDataset(val_pairs, IMG_SIZE, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    print(f"Data loaded: Train={len(train_pairs)}, Val={len(val_pairs)}, Test={len(test_pairs)}")
except Exception as e:
    print(f"데이터 로드 실패: {e}")

# ----------------- 모델 초기화 -----------------
model = SmoRestorLike(base_dim=48, num_blocks=4, heads=4, window_size=8).to(device)
criterion = DesmokeLoss(LossWeights()).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_psnr = -1.0
history = {"train_loss": [], "val_psnr": [], "val_ssim": [], "val_fade": [], "val_ciede": []}

# ----------------- 학습 루프 -----------------
for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*20} Epoch {epoch}/{EPOCHS} {'='*20}")
    
    # Train
    train_stats = train_one_epoch(model, train_loader, criterion, optimizer, epoch)
    history["train_loss"].append(train_stats['total'])
    
    # Evaluate
    val_stats = evaluate(model, val_loader, epoch)
    history["val_psnr"].append(val_stats['psnr'])
    history["val_ssim"].append(val_stats['ssim'])
    history["val_fade"].append(val_stats['fade'])
    history["val_ciede"].append(val_stats['ciede'])
    
    print(f"Train Loss: {train_stats['total']:.4f} | Val PSNR: {val_stats['psnr']:.2f} | Val SSIM: {val_stats['ssim']:.4f} | Val FADE: {val_stats['fade']:.4f} | Val CIEDE: {val_stats['ciede']:.4f}")
    
    # Save Best Model
    if val_stats["psnr"] > best_psnr:
        best_psnr = val_stats["psnr"]
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "best_psnr": best_psnr
        }, OUT_DIR / "best.pt")
        print("⭐ 최고 성능 모델 갱신 (Saved best.pt)")

## 8. 결과 그래프 분석

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(history['train_loss'], label='Train Loss (Total)', color='red')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.tick_params(axis='y', labelcolor='red')

ax2 = ax1.twinx()  
ax2.plot(history['val_psnr'], label='Val PSNR', color='blue', linestyle='--')
ax2.set_ylabel('PSNR')
ax2.tick_params(axis='y', labelcolor='blue')

fig.suptitle('Training Progress (Loss & PSNR)')
fig.legend(loc="center right", bbox_to_anchor=(0.85, 0.5))
plt.show()

## 9. 실시간 영상 테스트 및 FPS 벤치마크 (Real-time Surgical Desmoking)
학습된 모델을 실제 내시경 수술 영상(`assets/capture1.avi`)에 적용해 봅니다.
OpenCV를 이용하여 프레임 단위로 연기를 제거하고 결과 영상을 저장하며 처리 속도(FPS)를 측정합니다.

In [ ]:
import cv2
import time
import numpy as np

def process_video(model, video_path, output_path, img_size=(256, 512), device='cuda'):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[Error] 영상을 열 수 없습니다: {video_path}")
        return
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30.0 # 기본값 fallback
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # 결과물 저장을 위한 VideoWriter 설정 (원본 비율 유지 혹은 모델 비율로 조정 가능)
    # 원본과 복원 결과를 가로로 이어붙인 형태로 저장합니다.
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (img_size[1]*2, img_size[0]))
    
    model.eval()
    times = []
    print(f"\n영상 처리 시작: {video_path}")
    pbar = tqdm(total=total_frames, desc="Processing Video")
    
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
                
            # 1. 프레임 전처리 (BGR to RGB -> Resize -> Tensor)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            resized_frame = cv2.resize(rgb_frame, (img_size[1], img_size[0]))
            
            tensor_frame = transforms.ToTensor()(resized_frame).unsqueeze(0).to(device)
            
            # 2. 모델 추론 & 시간 측정
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            
            pred = model(tensor_frame)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000.0) # ms로 저장
            
            # 3. 후처리 및 병합 (Tensor -> numpy -> BGR)
            pred_np = pred[0].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
            pred_bgr = cv2.cvtColor((pred_np * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            
            orig_bgr = cv2.cvtColor(resized_frame, cv2.COLOR_RGB2BGR)
            
            # 원본과 복원 결과를 가로로 이어붙임
            combined_frame = np.hstack((orig_bgr, pred_bgr))
            out.write(combined_frame)
            
            pbar.update(1)
            
    cap.release()
    out.release()
    pbar.close()
    
    # 4. 성능 벤치마크 결과 계산
    if len(times) > 0:
        # 처음 몇 프레임은 warmup이므로 제외
        valid_times = times[5:] if len(times) > 10 else times
        mean_ms = sum(valid_times) / len(valid_times)
        fps_model = 1000.0 / mean_ms
        
        print(f"\n처리 완료! 결과 영상이 저장되었습니다: {output_path}")
        print(f"\n[🚀 벤치마크 결과]")
        print(f"- 프레임당 평균 소요 시간: {mean_ms:.2f} ms")
        print(f"- 모델 추론 실시간 속도(FPS): {fps_model:.2f} FPS")

        if fps_model >= 30:
            print("\n✅ [평가] 30 FPS 이상 달성! 실제 수술 환경에서 리얼타임 디스모킹 적용이 가능한 우수한 속도입니다.")
        else:
            print("\n⚠️ [평가] 30 FPS 미만입니다. 리얼타임 적용을 위해서는 해상도를 낮추거나 모델 경량화(Pruning/Quantization)가 필요합니다.")
    else:
        print("처리된 프레임이 없습니다.")

# 학습이 끝난 후 베스트 가중치를 다시 로드하여 영상 테스트 실행
try:
    best_ckpt = torch.load(OUT_DIR / "best.pt", map_location=device)
    model.load_state_dict(best_ckpt["model"])
    print("\n최고 성능 모델 가중치를 성공적으로 불러왔습니다.")
    
    video_in = "assets/capture1.avi"
    video_out = str(OUT_DIR / "capture1_desmoked_result.mp4")
    
    # 영상 처리 및 FPS 벤치마크 실행
    process_video(model, video_in, video_out, img_size=IMG_SIZE, device=device)

except FileNotFoundError:
    print("\n[안내] 아직 학습된 가중치(best.pt)가 없거나 영상을 찾을 수 없습니다. 학습을 먼저 완료해주세요.")
except Exception as e:
    print(f"\n영상 처리 중 오류 발생: {e}")